In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import models, transforms, datasets
from torch.utils.data import DataLoader, Subset, Dataset
import numpy as np
import pandas as pd
from sklearn.neighbors import NearestNeighbors

# --- Configuration ---
device = 'cuda' if torch.cuda.is_available() else 'cpu'
batch_size = 128
proxy_batch = 256
epochs = 30
warmup_epochs = 5
seed = 42
torch.manual_seed(seed)
np.random.seed(seed)

# ============================================================
# Dataset Loading (WITH EXACT STATISTICS)
# ============================================================
class StandardizedDataset(Dataset):
    def __init__(self, base):
        self.base = base

    def __len__(self):
        return len(self.base)

    def __getitem__(self, idx):
        x, y = self.base[idx]
        if isinstance(y, np.ndarray):
            y = int(y.item()) if y.size == 1 else int(y[0])
        elif torch.is_tensor(y):
            y = int(y.item())
        else:
            y = int(y)
        return x, y

def get_dataset(name):
    if name == 'fashion':
        tfm = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
        train = datasets.FashionMNIST(root='./data', train=True, transform=tfm, download=True)
        test = datasets.FashionMNIST(root='./data', train=False, transform=tfm, download=True)
        return StandardizedDataset(train), StandardizedDataset(test), 1, 10

    elif name == 'cifar100':
        # Applied the exact CIFAR-100 RGB statistics here
        tfm = transforms.Compose([
            transforms.ToTensor(), 
            transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
        ])
        train = datasets.CIFAR100(root='./data', train=True, transform=tfm, download=True)
        test = datasets.CIFAR100(root='./data', train=False, transform=tfm, download=True)
        return StandardizedDataset(train), StandardizedDataset(test), 3, 100
    else:
        raise ValueError(f"Unknown dataset: {name}")

# ============================================================
# Model & Feature Extraction (3x3 Kernel Fix)
# ============================================================
def get_model(in_channels, num_classes):
    model = models.resnet18(weights=None)
    model.conv1 = nn.Conv2d(in_channels, 64, kernel_size=3, stride=1, padding=1, bias=False)
    model.maxpool = nn.Identity()
    model.fc = nn.Linear(model.fc.in_features, num_classes)
    return model.to(device)

class EmbeddingExtractor:
    def __init__(self, model):
        self.features = None
        model.avgpool.register_forward_hook(self._hook)

    def _hook(self, module, inp, out):
        self.features = out.flatten(1).detach()

# ============================================================
# Proxy Scoring (MULTI-THREADED)
# ============================================================
def compute_proxy_signals(train_dataset, in_channels, num_classes, n_train):
    proxy = get_model(in_channels, num_classes)
    extractor = EmbeddingExtractor(proxy)
    opt = optim.Adam(proxy.parameters(), lr=0.001)
    criterion = nn.CrossEntropyLoss(reduction='none')
    
    # Speedup: num_workers=2 and pin_memory=True
    loader = DataLoader(train_dataset, batch_size=proxy_batch, shuffle=False, num_workers=2, pin_memory=True)

    proxy.train()
    for epoch in range(warmup_epochs):
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            out = proxy(x)
            loss = criterion(out, y)
            opt.zero_grad()
            loss.mean().backward()
            opt.step()
        print(f"    Proxy warm-up epoch {epoch + 1}/{warmup_epochs} done")

    proxy.eval()
    embeddings = np.zeros((n_train, 512), dtype=np.float32)
    el2n_scores = np.zeros(n_train, dtype=np.float32)
    eye = torch.eye(num_classes, device=device)

    with torch.no_grad():
        for i, (x, y) in enumerate(loader):
            x, y = x.to(device), y.to(device)
            out = proxy(x)
            emb = extractor.features
            probs = torch.softmax(out, dim=1)
            err = probs - eye[y]

            start = i * proxy_batch
            end = start + x.size(0)
            embeddings[start:end] = emb.cpu().numpy()
            el2n_scores[start:end] = torch.norm(err, dim=1).cpu().numpy()

    return embeddings, el2n_scores

# ============================================================
# HYBRID PPR-Coreset Selection (PRECOMPUTED FOR SPEED)
# ============================================================
def precompute_ppr_graph(embeddings, el2n_scores, k_neighbors=10, alpha=0.15):
    """Builds the K-NN graph and calculates PageRank once per dataset"""
    # Speedup: n_jobs=-1 uses all available Kaggle CPU cores
    nn_model = NearestNeighbors(n_neighbors=k_neighbors, metric='cosine', n_jobs=-1).fit(embeddings)
    adj = nn_model.kneighbors_graph(embeddings, mode='connectivity')

    row_sums = np.array(adj.sum(axis=1)).flatten()
    row_sums[row_sums == 0] = 1
    inv_row_sums = 1.0 / row_sums
    P = adj.multiply(inv_row_sums[:, None])

    v = el2n_scores / (el2n_scores.sum() + 1e-8)
    pi = v.copy()
    for _ in range(20):
        pi = (1 - alpha) * v + alpha * (P.T @ pi)

    return pi, adj.tocsr()

def select_hybrid_ppr(n, pi, adj_csr, num_samples, hybrid_ratio=0.5):
    """Runs only the greedy submodular selection step dynamically"""
    if num_samples >= n:
        return np.arange(n)

    num_random = int(num_samples * hybrid_ratio)
    num_ppr = num_samples - num_random
    random_indices = np.random.choice(n, num_random, replace=False)
    
    if num_ppr == 0:
        return random_indices

    selected_ppr = []
    current_gain = pi.copy()
    current_gain[random_indices] = -np.inf 

    for _ in range(num_ppr):
        idx = int(np.argmax(current_gain))
        selected_ppr.append(idx)
        neighbors = adj_csr[idx].indices
        valid_neighbors = neighbors[current_gain[neighbors] != -np.inf]
        if len(valid_neighbors) > 0:
            current_gain[valid_neighbors] *= 0.85 
        current_gain[idx] = -np.inf

    return np.concatenate([random_indices, np.array(selected_ppr)]).astype(int)

# ============================================================
# Training & Evaluation
# ============================================================
def train_and_eval(train_dataset, test_dataset, indices, in_channels, num_classes):
    # Speedup: num_workers=2 and pin_memory=True
    train_loader = DataLoader(Subset(train_dataset, indices), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False, num_workers=2, pin_memory=True)
    
    model = get_model(in_channels, num_classes)
    opt = optim.Adam(model.parameters(), lr=0.001)
    ce = nn.CrossEntropyLoss()

    model.train()
    for _ in range(epochs):
        for x, y in train_loader:
            opt.zero_grad()
            ce(model(x.to(device)), y.to(device)).backward()
            opt.step()

    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in test_loader:
            correct += (model(x.to(device)).argmax(1) == y.to(device)).sum().item()
    return correct / len(test_dataset)

# ============================================================
# Main Loop - PART 2
# ============================================================
results = []
datasets_names = ['fashion', 'cifar100']

for ds_name in datasets_names:
    print(f"\n=== Dataset: {ds_name} ===")
    train_dataset, test_dataset, in_channels, num_classes = get_dataset(ds_name)
    n_train = len(train_dataset)
    print(f"  n_train={n_train}, in_channels={in_channels}, num_classes={num_classes}")

    print("  Computing proxy signals (embeddings & EL2N)...")
    embeddings, el2n_scores = compute_proxy_signals(
        train_dataset, in_channels, num_classes, n_train
    )

    print("  Precomputing PPR Graph (Doing this ONCE to save time)...")
    pi, adj_csr = precompute_ppr_graph(embeddings, el2n_scores)

    for pct in [100, 80, 50, 25, 5]:
        num_samples = int(n_train * (pct / 100))
        
        # Run HYBRID PPR Selection (Fast)
        indices = select_hybrid_ppr(n_train, pi, adj_csr, num_samples)
        
        # Train and Evaluate
        acc = train_and_eval(train_dataset, test_dataset, indices, in_channels, num_classes)

        results.append({
            'Dataset': ds_name, 
            'Method': 'Hybrid-PPR', 
            'Percentage': pct,
            'NumSamples': len(indices), 
            'TestAcc': acc
        })
        
        print(f"  [{ds_name}] Method: Hybrid-PPR | Pct: {pct:3d}% | "
              f"N: {len(indices):6d} | Acc: {acc:.4f}")

    pd.DataFrame(results).to_csv('hybrid_ppr_results_part2.csv', index=False)

print("\nExperiment complete. Results saved to hybrid_ppr_results_part2.csv")


=== Dataset: fashion ===


100%|██████████| 26.4M/26.4M [00:01<00:00, 19.2MB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 306kB/s]
100%|██████████| 4.42M/4.42M [00:00<00:00, 5.70MB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 25.1MB/s]


  n_train=60000, in_channels=1, num_classes=10
  Computing proxy signals (embeddings & EL2N)...
    Proxy warm-up epoch 1/5 done
    Proxy warm-up epoch 2/5 done
    Proxy warm-up epoch 3/5 done
    Proxy warm-up epoch 4/5 done
    Proxy warm-up epoch 5/5 done
  Precomputing PPR Graph (Doing this ONCE to save time)...
  [fashion] Method: Hybrid-PPR | Pct: 100% | N:  60000 | Acc: 0.9277
  [fashion] Method: Hybrid-PPR | Pct:  80% | N:  48000 | Acc: 0.9295
  [fashion] Method: Hybrid-PPR | Pct:  50% | N:  30000 | Acc: 0.9173
  [fashion] Method: Hybrid-PPR | Pct:  25% | N:  15000 | Acc: 0.8713
  [fashion] Method: Hybrid-PPR | Pct:   5% | N:   3000 | Acc: 0.7575

=== Dataset: cifar100 ===


100%|██████████| 169M/169M [33:10<00:00, 84.9kB/s]


  n_train=50000, in_channels=3, num_classes=100
  Computing proxy signals (embeddings & EL2N)...
    Proxy warm-up epoch 1/5 done
    Proxy warm-up epoch 2/5 done
    Proxy warm-up epoch 3/5 done
    Proxy warm-up epoch 4/5 done
    Proxy warm-up epoch 5/5 done
  Precomputing PPR Graph (Doing this ONCE to save time)...
  [cifar100] Method: Hybrid-PPR | Pct: 100% | N:  50000 | Acc: 0.5421
  [cifar100] Method: Hybrid-PPR | Pct:  80% | N:  40000 | Acc: 0.5179
  [cifar100] Method: Hybrid-PPR | Pct:  50% | N:  25000 | Acc: 0.4591
  [cifar100] Method: Hybrid-PPR | Pct:  25% | N:  12500 | Acc: 0.3418
  [cifar100] Method: Hybrid-PPR | Pct:   5% | N:   2500 | Acc: 0.1293

Experiment complete. Results saved to hybrid_ppr_results_part2.csv
